In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import os

# --- Load cleaned data ---
df = pd.read_csv('data/cleaned_churn_data.csv')

# Separate features, target, and customerID (keep ID aside)
customer_ids = df['customerID']
X = df.drop(['Churn', 'customerID'], axis=1)
y = df['Churn']

print("Feature shape:", X.shape)
print("Target distribution:\n", y.value_counts(normalize=True))

# --- Train/Test Split (80/20, stratify to keep churn ratio same in both sets) ---
X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, customer_ids, test_size=0.2, random_state=42, stratify=y
)

print("\nTrain shape:", X_train.shape, "Test shape:", X_test.shape)

# --- Scale features (Logistic Regression needs this, Random Forest doesn't but no harm) ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# =========================
# MODEL 1: Logistic Regression
# =========================
log_model = LogisticRegression(max_iter=1000, random_state=42)
log_model.fit(X_train_scaled, y_train)
log_preds = log_model.predict(X_test_scaled)

print("\n===== Logistic Regression =====")
print("Accuracy:", accuracy_score(y_test, log_preds))
print("Precision:", precision_score(y_test, log_preds))
print("Recall:", recall_score(y_test, log_preds))
print("F1 Score:", f1_score(y_test, log_preds))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, log_preds))
print("\nClassification Report:\n", classification_report(y_test, log_preds))

# =========================
# MODEL 2: Random Forest
# =========================
rf_model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)  # RF doesn't need scaling
rf_preds = rf_model.predict(X_test)

print("\n===== Random Forest =====")
print("Accuracy:", accuracy_score(y_test, rf_preds))
print("Precision:", precision_score(y_test, rf_preds))
print("Recall:", recall_score(y_test, rf_preds))
print("F1 Score:", f1_score(y_test, rf_preds))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, rf_preds))
print("\nClassification Report:\n", classification_report(y_test, rf_preds))

# =========================
# Feature Importance (Random Forest) — useful for insights later
# =========================
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 Important Features:\n", feature_importance.head(10))

# =========================
# Generate churn_probability for FULL dataset (for Power BI)
# =========================
X_scaled_full = scaler.transform(X)
churn_probabilities = log_model.predict_proba(X_scaled_full)[:, 1]  # probability of Churn=1

df['churn_probability'] = churn_probabilities
df['customerID'] = customer_ids.values  # ensure it's there

# Save final export for Power BI
os.makedirs('data', exist_ok=True)
export_cols = ['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn', 'churn_probability'] + \
              [c for c in df.columns if c not in ['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn', 'churn_probability']]
df[export_cols].to_csv('data/churn_predictions_for_powerbi.csv', index=False)

print("\n Exported: data/churn_predictions_for_powerbi.csv")
feature_importance.to_csv('data/feature_importance.csv', index=False)
print(" Exported: data/feature_importance.csv")

Feature shape: (7043, 35)
Target distribution:
 Churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64

Train shape: (5634, 35) Test shape: (1409, 35)

===== Logistic Regression =====
Accuracy: 0.8090844570617459
Precision: 0.6755852842809364
Recall: 0.5401069518716578
F1 Score: 0.600297176820208

Confusion Matrix:
 [[938  97]
 [172 202]]

Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.91      0.87      1035
           1       0.68      0.54      0.60       374

    accuracy                           0.81      1409
   macro avg       0.76      0.72      0.74      1409
weighted avg       0.80      0.81      0.80      1409


===== Random Forest =====
Accuracy: 0.7998580553584103
Precision: 0.6554054054054054
Recall: 0.5187165775401069
F1 Score: 0.5791044776119403

Confusion Matrix:
 [[933 102]
 [180 194]]

Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.9

In [3]:
# pip install imbalanced-learn (agar error aaye "module not found" toh terminal mein: pip install imbalanced-learn)

from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

log_model_smote = LogisticRegression(max_iter=1000, random_state=42)
log_model_smote.fit(X_train_smote, y_train_smote)
smote_preds = log_model_smote.predict(X_test_scaled)

print("===== Logistic Regression + SMOTE =====")
print("Accuracy:", accuracy_score(y_test, smote_preds))
print("Precision:", precision_score(y_test, smote_preds))
print("Recall:", recall_score(y_test, smote_preds))
print("F1 Score:", f1_score(y_test, smote_preds))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, smote_preds))

===== Logistic Regression + SMOTE =====
Accuracy: 0.7352732434350603
Precision: 0.5008403361344538
Recall: 0.7967914438502673
F1 Score: 0.6150670794633643

Confusion Matrix:
 [[738 297]
 [ 76 298]]


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


In [4]:
# Generate churn_probability using SMOTE-trained model (FINAL MODEL)

# Scale full dataset (same scaler used before)
X_scaled_full = scaler.transform(X)

# Use SMOTE model for probabilities (better recall, business-justified)
churn_probabilities_smote = log_model_smote.predict_proba(X_scaled_full)[:, 1]

df['churn_probability'] = churn_probabilities_smote
df['customerID'] = customer_ids.values

# Save final export for Power BI
export_cols = ['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn', 'churn_probability'] + \
              [c for c in df.columns if c not in ['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn', 'churn_probability']]

df[export_cols].to_csv('data/churn_predictions_for_powerbi.csv', index=False)

print(" Final export done (SMOTE model): data/churn_predictions_for_powerbi.csv")
print(df[['customerID', 'Churn', 'churn_probability']].head(10))

✅ Final export done (SMOTE model): data/churn_predictions_for_powerbi.csv
   customerID  Churn  churn_probability
0  7590-VHVEG      0           0.893104
1  5575-GNVDE      0           0.075676
2  3668-QPYBK      1           0.544434
3  7795-CFOCW      0           0.067582
4  9237-HQITU      1           0.875835
5  9305-CDSKC      1           0.918150
6  1452-KIOVK      0           0.665621
7  6713-OKOMC      0           0.431688
8  7892-POOKP      1           0.803970
9  6388-TABGU      0           0.038616


In [5]:
#Revenue at Risk
df['revenue_at_risk'] = df['churn_probability'] * df['MonthlyCharges'] * 12  # annualized

total_revenue_at_risk = df['revenue_at_risk'].sum()
print(f" Total Annualized Revenue at Risk: ${total_revenue_at_risk:,.2f}")
print(f" Avg Revenue at Risk per Customer: ${df['revenue_at_risk'].mean():,.2f}")

 Total Annualized Revenue at Risk: $2,508,238.92
 Avg Revenue at Risk per Customer: $356.13


In [15]:
#ROC-AUC Scores
from sklearn.metrics import roc_auc_score

auc_log = roc_auc_score(y_test, log_model.predict_proba(X_test_scaled)[:, 1])
auc_smote = roc_auc_score(y_test, log_model_smote.predict_proba(X_test_scaled)[:, 1])
auc_rf = roc_auc_score(y_test, rf_model.predict_proba(X_test)[:, 1])

print(f"ROC-AUC Scores:")
print(f"Logistic Regression (base): {auc_log:.4f}")
print(f"Logistic Regression (SMOTE): {auc_smote:.4f}")
print(f"Random Forest: {auc_rf:.4f}")

ROC-AUC Scores:
Logistic Regression (base): 0.8480
Logistic Regression (SMOTE): 0.8458
Random Forest: 0.8420


In [16]:
#Customer Segmentation (K-Means)
from sklearn.cluster import KMeans

cluster_features = df[['tenure', 'MonthlyCharges', 'churn_probability']]
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['customer_segment'] = kmeans.fit_predict(cluster_features)

segment_summary = df.groupby('customer_segment').agg(
    avg_tenure=('tenure', 'mean'),
    avg_monthly_charges=('MonthlyCharges', 'mean'),
    avg_churn_prob=('churn_probability', 'mean'),
    count=('customerID', 'count')
).round(2)

print("Segment Summary:")
print(segment_summary)

Segment Summary:
                  avg_tenure  avg_monthly_charges  avg_churn_prob  count
customer_segment                                                        
0                      54.17                33.96            0.09   1153
1                      14.77                81.09            0.67   2186
2                      58.58                93.28            0.30   1959
3                      10.59                32.64            0.39   1745


In [17]:
#Final Export
export_cols = ['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn', 
               'churn_probability', 'revenue_at_risk', 'customer_segment'] + \
              [c for c in df.columns if c not in ['customerID', 'tenure', 'MonthlyCharges', 
               'TotalCharges', 'Churn', 'churn_probability', 'revenue_at_risk', 'customer_segment']]

df[export_cols].to_csv('data/churn_predictions_for_powerbi.csv', index=False)
print(" Final export updated with revenue_at_risk and customer_segment")

 Final export updated with revenue_at_risk and customer_segment
